In [4]:
import pandas as pd 
from IPython.display import display

# 读取数据
movies = pd.read_csv('movies.csv')
display(movies.head(10))

# 创建 genre 矩阵（one-hot 编码）
genre_matrix = movies['genres'].str.get_dummies(sep='|')

# 显示矩阵
display(genre_matrix.head())

# 检查类型
print(f"genre_matrix 类型: {type(genre_matrix)}")
print(f"genre_matrix 形状: {genre_matrix.shape}")

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


genre_matrix 类型: <class 'pandas.core.frame.DataFrame'>
genre_matrix 形状: (9125, 20)


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim=cosine_similarity(genre_matrix)
print("相似度矩阵形状:",cosine_sim.shape)

相似度矩阵形状: (9125, 9125)


In [24]:
def get_recommendations(title):
    try:
        idx = movies[movies['title'] == title].index[0]
    except:
        return "电影库没找到这部电影，请检查拼写（包含年份）"
    
    # 获取相似度分数
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:6]  # 排除自己，取前5个
    
    # 提取电影索引和分数
    movie_indices = [i[0] for i in sim_scores]
    movie_scores = [i[1] for i in sim_scores]
    
    # 创建结果DataFrame
    recommendations = movies['title'].iloc[movie_indices].reset_index(drop=True)
    scores_df = pd.DataFrame(movie_scores, columns=['cosine_similarity'])
    
    # 合并显示
    result = pd.concat([recommendations, scores_df], axis=1)
    result.columns = ['推荐电影', '相似度分数']
    
    return result

In [25]:
test_movie=movies['title'].iloc[9]
print(f"测试电影:{test_movie}")
recommendations=get_recommendations(test_movie)
print(recommendations)

测试电影:GoldenEye (1995)
                        推荐电影  相似度分数
0        Broken Arrow (1996)    1.0
1         Cliffhanger (1993)    1.0
2  Executive Decision (1996)    1.0
3  Surviving the Game (1994)    1.0
4           Rock, The (1996)    1.0


In [22]:
print(movies[movies['title']==test_movie])

   movieId             title                     genres
9       10  GoldenEye (1995)  Action|Adventure|Thriller


In [26]:
display(movies.head(491))

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
486,542,Son in Law (1993),Comedy|Drama|Romance
487,543,So I Married an Axe Murderer (1993),Comedy|Romance|Thriller
488,544,Striking Distance (1993),Action|Crime
489,546,Super Mario Bros. (1993),Action|Adventure|Children|Comedy|Fantasy|Sci-Fi


In [6]:
#基于协同过滤的个性化推荐

In [7]:
import pandas as pd
from IPython.display import display

In [11]:
#读取文件
ratings=pd.read_csv('ratings.csv')
display(ratings.head(20))

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205
5,1,1263,2.0,1260759151
6,1,1287,2.0,1260759187
7,1,1293,2.0,1260759148
8,1,1339,3.5,1260759125
9,1,1343,2.0,1260759131


In [18]:
#将表格转换成矩阵
user_movie_matrix=ratings.pivot(index='userId',columns='movieId',values='rating')

In [19]:
#矩阵的列是电影
user_movie_matrix=user_movie_matrix.dropna(thresh=10,axis=1)

In [33]:
#矩阵的行是用户
user_movie_matrix=user_movie_matrix.dropna(thresh=20,axis=0)

In [34]:
user_movie_matrix_filled=user_movie_matrix.fillna(0)
print("过滤后的矩阵形状",user_movie_matrix_filled.shape)
print("前五行数据",user_movie_matrix_filled.head(5))

过滤后的矩阵形状 (625, 1303)
前五行数据 movieId  1       2       3       5       6       7       9       10      \
userId                                                                    
1           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2           0.0     0.0     0.0     0.0     0.0     0.0     0.0     4.0   
3           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
4           0.0     0.0     0.0     0.0     0.0     0.0     0.0     4.0   
5           0.0     0.0     4.0     0.0     0.0     0.0     0.0     0.0   

movieId  11      14      ...  112552  112556  112852  115617  115713  116797  \
userId                   ...                                                   
1           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
2           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
3           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
4           0.0     0.0  ...     0.0     0.0   

In [35]:
#计算用户相似度
from sklearn.metrics.pairwise import cosine_similarity
user_sim=cosine_similarity(user_movie_matrix_filled)
#转换成Dataframe，给定标签便于索引
user_sim_df=pd.DataFrame(user_sim,
                         index=user_movie_matrix.index,
                         columns=user_movie_matrix.index)
print("用户相似度矩阵（前五名用户）:")
print(user_sim_df.head(5))

用户相似度矩阵（前五名用户）:
userId       1         2         3         4         5         6         7    \
userId                                                                         
1       1.000000  0.000000  0.000000  0.082005  0.016818  0.000000  0.084864   
2       0.000000  1.000000  0.138650  0.135217  0.107129  0.000000  0.222712   
3       0.000000  0.138650  1.000000  0.097006  0.163536  0.066503  0.168921   
4       0.082005  0.135217  0.097006  1.000000  0.143843  0.089036  0.356147   
5       0.016818  0.107129  0.163536  0.143843  1.000000  0.064773  0.097007   

userId       8         9         10   ...       660       661       662  \
userId                                ...                                 
1       0.000000  0.013224  0.000000  ...  0.000000  0.033301  0.000000   
2       0.119908  0.120617  0.049284  ...  0.021465  0.026412  0.502227   
3       0.276285  0.149435  0.136558  ...  0.063980  0.094250  0.177110   
4       0.215543  0.034483  0.166662  ...  0.071

In [36]:
def get_user_recommendations(user_id, top_n=5):
    # 1. 找到和该用户最像的前 10 个人 (用方括号，变量名对齐)
    similar_users = user_sim_df[user_id].sort_values(ascending=False)[1:11].index
    
    # 2. 看看这些“邻居”都评价过哪些电影 
    neighbor_ratings = ratings[ratings['userId'].isin(similar_users)]
    
    # 3. 排除掉该用户已经看过的电影
    user_watched = ratings[ratings['userId'] == user_id]['movieId']
    recommendations = neighbor_ratings[~neighbor_ratings['movieId'].isin(user_watched)]
    
    # 4. 计算这些电影在邻居中的平均分，并排序
    res = recommendations.groupby('movieId')['rating'].mean().sort_values(ascending=False).head(top_n)
    
    # 5. 联动 movies 表展示结果
    return movies[movies['movieId'].isin(res.index)][['title', 'genres']]

In [37]:
print(f"为用户{user_movie_matrix.index[0]}推荐的电影列表:")
get_user_recommendations(user_movie_matrix.index[0])

为用户1推荐的电影列表:


,title,genres
64,Friday (1995),Comedy
1643,"Muppet Christmas Carol, The (1992)",Children|Comedy|Musical
1662,Steamboat Willie (1928),Animation|Children|Comedy|Musical
2635,They Might Be Giants (1971),Comedy|Mystery|Romance
7194,Land of Silence and Darkness (Land des Schweig...,Documentary
